In [1]:
#%% 
! pip install transformers==4.49.0 keybert bertviz lime

In [2]:
#%%
# Constants
MODEL_PATH = "huggingface/CodeBERTa-language-id"
js_code = """
import mod189 from './mod189';
var value=mod189+1;
export default value;
"""
python_code = """
def ciao():
    end = idk() + random + "return_"
    return
"""
CODE_TO_TEST = python_code


#%% 

In [3]:
# Loading the model and tokenizer
from transformers import RobertaTokenizer, RobertaForSequenceClassification

from transformers import TextClassificationPipeline

model = RobertaForSequenceClassification.from_pretrained(MODEL_PATH, output_attentions=True)
tokenizer = RobertaTokenizer.from_pretrained(MODEL_PATH)

pipeline_ = TextClassificationPipeline(
    model=model,
    tokenizer=tokenizer
)


#%% 
# Base Testing


# not working
# inputs = tokenizer(CODE_TO_TEST)
# print(inputs["input_ids"])
# output = model(inputs)[0]
# language_id = output.argmax()
# print(language_id)

print(pipeline_(CODE_TO_TEST))
#%% 

Some weights of the model checkpoint at huggingface/CodeBERTa-language-id were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
RobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementat

[{'label': 'ruby', 'score': 0.577468991279602}]


In [7]:
# trying keyBert

from keybert import KeyBERT
from transformers import pipeline
kb_model = KeyBERT(model=model)

extract_embeddings = pipeline("feature-extraction", MODEL_PATH)
embeddings = extract_embeddings(CODE_TO_TEST)
print(len(embeddings), len(embeddings[0]), len(embeddings[0][0]))

kb_model.extract_keywords(CODE_TO_TEST, highlight=True, use_maxsum= False)
#%%

Device set to use cpu


1 25 768


def ciao end idk random return_ return

[('return_', 0.5227),
 ('def', 0.5063),
 ('return', 0.4723),
 ('random', 0.353),
 ('idk', 0.2803)]

In [5]:
# trying Bertviz

from bertviz import head_view, model_view, neuron_view

inputs = tokenizer.encode_plus(CODE_TO_TEST, return_tensors='pt')
outputs = model(inputs["input_ids"])
# print(outputs)
attention = model(inputs["input_ids"])[-1]
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())
# print(tokenizer.decode(inputs[0]))


# head_view(attention, tokens)
model_view(attention, tokens)


# %%

<IPython.core.display.Javascript object>

In [6]:
# Trying LIME
import lime, torch
from lime.lime_text import LimeTextExplainer
class_names = [
    "go",
    "java",
    "javascript",
    "php",
    "python",
    "ruby",
]

def prediction(texts):
    outputs = model(**tokenizer(texts, return_tensors="pt", padding=True))
    tensor_logits = outputs[0]
    # print(tensor_logits)
    probs = torch.nn.functional.softmax(tensor_logits).detach().numpy()
    # print(probs)
    return probs

# print(tokenizer(CODE_TO_TEST, return_tensors='pt', padding=True))

explainer = LimeTextExplainer(class_names=class_names)

explaination = explainer.explain_instance(CODE_TO_TEST, prediction, num_features=15, num_samples=1000, top_labels=2)

# print(explaination.available_labels())
explaination.show_in_notebook(text=CODE_TO_TEST)

#%%

/tmp/ipykernel_1812/2916665547.py:17: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probs = torch.nn.functional.softmax(tensor_logits).detach().numpy()
